## 0 · Preparación
## Setup de Infraestructura (Ejecución Única)

Este notebook prepara la infraestructura base del proyecto ejecutando de forma desacoplada el script **`../sql/00_setup_ddl.sql`**.

* **Parametrización con Widgets:** Utiliza los parámetros `catalogo` (`workspace`) y `esquema` (`demo_incremental`) para inyectar dinámicamente el entorno sobre las variables `{catalogo}` y `{esquema}` del archivo SQL[cite: 8].
* **Objetos creados:** Reinicia el esquema (`DROP/CREATE SCHEMA`), crea el volumen de llegada (`archivos`) y construye las tablas Delta base (`bronce_clientes`, `clientes_scd1`, `clientes_scd2` y `oro_clientes_por_ciudad`)[cite: 8].
* **Fuera de la orquestación (Jobs):** Al contener sentencias destructivas de reinicio (`CASCADE`), este notebook **no forma parte del Databricks Job automatizado**; se ejecuta manualmente una sola vez antes de activar el flujo incremental (`Bronce → Plata → Oro`).

**Qué esperar:** Mensaje de confirmación `✅ Infraestructura creada exitosamente en workspace.demo_incremental`[cite: 8].

**Qué esperar:** `OK`.

In [0]:
# 1. Definir widgets para parametrizar entorno (permite sobrescribir desde Jobs)
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("esquema", "demo_incremental", "Esquema")

catalogo = dbutils.widgets.get("catalogo")
esquema  = dbutils.widgets.get("esquema")

# 2. Leer el archivo SQL externo usando ruta relativa desde /notebook/ hacia /sql/
ruta_sql = "../sql/00_setup_ddl.sql"
with open(ruta_sql, "r", encoding="utf-8") as f:
    script_sql = f.read()

# 3. Reemplazar parámetros dinámicos
script_sql = (
    script_sql
    .replace("{catalogo}", catalogo)
    .replace("{esquema}", esquema)
)

# 4. Ejecutar sentencia por sentencia (Spark SQL ejecuta una instrucción por llamada)
sentencias = [s.strip() for s in script_sql.split(";") if s.strip()]

for i, sentencia in enumerate(sentencias, start=1):
    # Ignorar bloques que solo tengan comentarios
    lineas_codigo = [l for l in sentencia.splitlines() if not l.strip().startswith("--") and l.strip()]
    if lineas_codigo:
        print(f"Ejecutando sentencia {i}...")
        spark.sql(sentencia)

print(f"✅ Infraestructura creada exitosamente en {catalogo}.{esquema}")